Ce Notebook est inutile les resultats qu'il apporte ne sont pas concluants...

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from tools import line
from pathlib import Path

FIG_DIR = Path(r'..\reports\figures\airbnb')
FIG_WIDTH, FIG_HEIGHT = 1600, 500

# Chargement

In [3]:
# Comptabilité
df0 = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])
# DF Loyers
loyers = df0.query("code_compte == '70600000'")
# Colonne "canal"
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

# AirBNB
airbnb = pd.read_csv(r"..\data\processed\airbnb.csv", sep=';', decimal=',', parse_dates=['mois'])
airbnb['mois'] = airbnb['mois'].dt.to_period('M')

# Préparation

In [4]:
ca_airbnb = loyers.query("canal == 'AIRBNB'")
ca_airbnb = ca_airbnb.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum()
ca_airbnb = ca_airbnb.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_airbnb['periode'] = ca_airbnb['date_écriture'].astype(str)

# Merge

In [5]:
df_merge = ca_airbnb.merge(airbnb, left_on='date_écriture', right_on='mois', how='left')

In [6]:
df_merge.head()

,date_écriture,crédit_euro,débit_euro,ca_net,periode,mois,consultations,réservations,taux_de_réservation,period_label,année,var_consultations_pct,var_consultations_pct_annuel,var_resa_pct_annuel,var_resa_pts_annuel
0,2024-07,1037.27,0.0,1037.27,2024-07,2024-07,680,17,2.5,Juillet 2024,2024,NaN,NaN,NaN,NaN
1,2024-08,3005.59,0.0,3005.59,2024-08,2024-08,352,8,2.3,Août 2024,2024,-48.235294,NaN,NaN,NaN
2,2024-09,1122.10,0.0,1122.10,2024-09,2024-09,442,9,2.0,Septembre 2024,2024,25.568182,NaN,NaN,NaN
3,2024-10,1332.25,0.0,1332.25,2024-10,2024-10,308,2,0.6,Octobre 2024,2024,-30.316742,NaN,NaN,NaN
4,2024-11,308.48,0.0,308.48,2024-11,2024-11,124,0,0.0,Novembre 2024,2024,-59.740260,NaN,NaN,NaN


# CA par Reservation

In [7]:
df = df_merge[['date_écriture', 'ca_net', 'réservations']]
df['ca_par_resa'] = df['ca_net'] / df['réservations']
df

,date_écriture,ca_net,réservations,ca_par_resa
0,2024-07,1037.27,17,61.015882
1,2024-08,3005.59,8,375.698750
2,2024-09,1122.10,9,124.677778
3,2024-10,1332.25,2,666.125000
4,2024-11,308.48,0,inf
5,2024-12,1578.10,5,315.620000
6,2025-01,169.66,5,33.932000
7,2025-02,221.72,5,44.344000
8,2025-03,865.67,15,57.711333
9,2025-04,1562.64,4,390.660000
